<a href="https://colab.research.google.com/github/mohanasudhashanmugam/DeepLearning/blob/main/Modified_train_theft_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!kaggle datasets download -d kipshidze/shoplifting-video-dataset
!unzip -q shoplifting-video-dataset.zip -d ./local_colab_storage

In [ ]:
!ls /content/local_colab_storage/normal | wc -l

In [ ]:
!ls /content/local_colab_storage/shoplifting | wc -l

In [ ]:
import cv2
import os
import numpy as np
import argparse
from imutils import paths
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
# construct the argument parser and parse the arguments
ap = argparse.ArgumentParser()

ap.add_argument("-d", "--dataset",
	default="/content/local_colab_storage/",
	help="path to input dataset")
args_namespace, unknown = ap.parse_known_args()


# Convert to a dictionary
args = vars(args_namespace)

In [ ]:
video_paths = list(paths.list_files(
    args["dataset"],
    validExts=video_extensions
))

video_labels = []

for video_path in video_paths:
    label = video_path.split(os.path.sep)[-2]
    video_labels.append(label)

print("Total videos:", len(video_paths))
print("Labels:", set(video_labels))

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(video_labels)

print("Classes:", label_encoder.classes_)
print("Encoded labels:", y)

In [ ]:
from sklearn.model_selection import train_test_split

train_paths, test_paths, train_labels, test_labels = train_test_split(
    video_paths,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training videos:", len(train_paths))
print("Testing videos:", len(test_paths))

In [ ]:
import tensorflow as tf

class VideoDataGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        video_paths,
        labels,
        batch_size=4,
        max_frames=32,
        resize_dim=(224, 224),
        shuffle=True
    ):
        self.video_paths = video_paths
        self.labels = labels
        self.batch_size = batch_size
        self.max_frames = max_frames
        self.resize_dim = resize_dim
        self.shuffle = shuffle

        self.indices = np.arange(len(self.video_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.video_paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def process_video(self, video_path):

        cap = cv2.VideoCapture(video_path)

        total_frames = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        if total_frames <= 0:
            cap.release()
            return np.zeros(
                (self.max_frames, *self.resize_dim, 3),
                dtype=np.float32
            )

        frame_indices = np.linspace(
            0,
            total_frames - 1,
            self.max_frames,
            dtype=int
        )

        video_frames = []

        for frame_idx in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                frame_idx
            )

            success, frame = cap.read()

            if not success:
                break

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            frame = cv2.resize(
                frame,
                self.resize_dim
            )

            frame = frame.astype(
                np.float32
            ) / 255.0

            video_frames.append(frame)

        cap.release()

        # Make sure every video has exactly 32 frames
        while len(video_frames) < self.max_frames:

            video_frames.append(
                video_frames[-1].copy()
                if video_frames
                else np.zeros(
                    (*self.resize_dim, 3),
                    dtype=np.float32
                )
            )

        return np.array(
            video_frames,
            dtype=np.float32
        )

    def __getitem__(self, index):

        start = index * self.batch_size
        end = min(
            start + self.batch_size,
            len(self.video_paths)
        )

        batch_indices = self.indices[start:end]

        X_batch = []
        y_batch = []

        for i in batch_indices:

            video = self.process_video(
                self.video_paths[i]
            )

            X_batch.append(video)
            y_batch.append(self.labels[i])

        X_batch = np.array(
            X_batch,
            dtype=np.float32
        )

        y_batch = tf.keras.utils.to_categorical(
            y_batch,
            num_classes=2
        )

        return X_batch, y_batch

In [ ]:
train_generator = VideoDataGenerator(
    train_paths,
    train_labels,
    batch_size=2,
    max_frames=32,
    resize_dim=(224, 224),
    shuffle=True
)

test_generator = VideoDataGenerator(
    test_paths,
    test_labels,
    batch_size=2,
    max_frames=32,
    resize_dim=(224, 224),
    shuffle=False
)

In [ ]:
X_batch, y_batch = train_generator[0]

print("Batch X shape:", X_batch.shape)
print("Batch y shape:", y_batch.shape)
print("Batch X dtype:", X_batch.dtype)
print("Batch X memory:",
      X_batch.nbytes / (1024**2),
      "MB")

In [ ]:

from tensorflow.keras import layers, models

aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomContrast(0.1)
])

# 1. Base CNN to extract features from a single frame
# We use MobileNetV2 because it is lightweight and fast

base_cnn = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_cnn.trainable = False  # Freeze weights as we don't destroy pre-trained weights by new training data during backpropagation

# Flatten the CNN output to a vector
pooling_layer = layers.GlobalAveragePooling2D()(base_cnn.output)
feature_extractor = models.Model(
    inputs=base_cnn.input,
    outputs=pooling_layer
)
# 2. Complete Video Model
video_input = layers.Input(
    shape=(32, 224, 224, 3)
)
# TimeDistributed applies the CNN to all frames individually
augmented_input = layers.TimeDistributed(aug)(video_input)
encoded_frames = layers.TimeDistributed(
    feature_extractor
)(augmented_input)

# LSTM tracks the movement across the timeline
x = layers.LSTM(
    64,
    dropout=0.5
)(encoded_frames)

x = layers.Dense(
    32,
    activation='relu'
)(x)

# # Output layer: Sigmoid activation for binary classification (0 or 1)
output = layers.Dense(
    1,
    activation='sigmoid'
)(x)

model = models.Model(
    inputs=video_input,
    outputs=output
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
## Run training for 30 to 50 epochs, but use an EarlyStopping callback.
#This tells Colab to keep training as long as the validation loss is improving, and automatically stop if it plateaus, saves time.

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=10,
    callbacks=[early_stop]
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Get raw probability predictions (numbers between 0.0 and 1.0)
predictions = model.predict(X_test)
print(predictions)

In [ ]:
# 2. Convert probabilities to binary choices: if > 0.5, it's Shoplifting (1), else Normal (0)
##binary_predictions = (predictions > 0.5).astype(int)

binary_predictions = np.argmax(predictions, axis=1)

# Convert y_test from 2 columns to 1 column
y_test_labels = np.argmax(y_test, axis=1)

print(binary_predictions)

In [ ]:
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test_labels, binary_predictions))

In [ ]:
print("\n--- Classification Report ---")
print(classification_report(y_test_labels, binary_predictions, target_names=['Normal', 'Shoplifting']))